# Chapter 6: Using a Neural Network to Fit the Data
### Artificial Neurons, Non-Linear Activations, the `torch.nn` Module, and Custom Subclassing

This companion notebook implements the complete pedagogical workflow of **Chapter 6** from *Deep Learning with PyTorch (2nd Edition)* by Eli Stevens, Luca Antiga, Thomas Viehmann, and Howard Huang.

#### Contents:
1. Setup & Compute Environment
2. Dataset Representation & The $N \times C_{\text{in}}$ Batch Mandate
3. Anatomy of an Artificial Neuron & Activation Functions Gallery
4. Affine Projections with `nn.Linear` & The Callable Rule
5. Assembling Deep Architectures with `nn.Sequential`
6. Parameter Inspection & Layer Naming with `OrderedDict`
7. Training Dynamics: Adaptive Optimization with Adam
8. Visualizing Non-Linear Interpolation & Expressivity
9. Advanced OOP Architecture: Subclassing `nn.Module` & Residual Blocks

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from collections import OrderedDict

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch Version: {torch.__version__}")
print(f"Active Compute Device: {device}")

## 1. Dataset Representation & The $N \times C_{\text{in}}$ Batch Mandate
In PyTorch, all `nn.Module` layers expect the **zeroth dimension** to represent the batch size ($N$). We reshape our 1D thermometer measurements of shape `(11,)` into 2D tensors of shape `(11, 1)` and normalize inputs ($t_{\text{un}} = 0.1 \times t_u$).

In [ ]:
# Empirical thermometer observations
t_c = [0.5, 14.0, 15.0, 28.0, 11.0, 8.0, 3.0, -4.0, 6.0, 13.0, 21.0]
t_u = [35.7, 55.9, 58.2, 81.9, 56.3, 48.9, 33.9, 21.8, 48.4, 60.4, 68.4]

# Allocate 32-bit float tensors
t_c = torch.tensor(t_c, dtype=torch.float32).unsqueeze(1)
t_u = torch.tensor(t_u, dtype=torch.float32).unsqueeze(1)
t_un = 0.1 * t_u

print(f"t_c shape:  {t_c.shape}")
print(f"t_u shape:  {t_u.shape}")
print(f"t_un shape: {t_un.shape}")

## 2. Anatomy of an Artificial Neuron & Activation Functions Gallery
A neuron executes an affine transformation followed by a non-linear activation:
$$o = \sigma(\mathbf{w}^T \mathbf{x} + b)$$
Let us evaluate a scalar neuron with $w=2, b=6$ under $\tanh$ activation across three sample inputs.

In [ ]:
def single_neuron(x, w=2.0, b=6.0):
    z = w * x + b
    return torch.tanh(torch.tensor(z, dtype=torch.float32)).item()

test_inputs = [18.0, -2.79, -10.0]
for x in test_inputs:
    out = single_neuron(x)
    print(f"Input x = {x:6.2f} -> Linear z = {2.0*x+6.0:6.2f} -> Tanh output o = {out:7.4f}")

### 2.1 Visualizing Core Activation Functions
We compare six foundational activations in `torch.nn` against the identity line $y = x$.

In [ ]:
x_vals = torch.linspace(-3, 3, 300)

fig, axes = plt.subplots(2, 3, figsize=(12, 7))
activations = [
    ("Tanh", nn.Tanh()),
    ("Hardtanh", nn.Hardtanh()),
    ("Sigmoid", nn.Sigmoid()),
    ("Softplus", nn.Softplus()),
    ("ReLU", nn.ReLU()),
    ("LeakyReLU", nn.LeakyReLU(negative_slope=0.1))
]

for ax, (name, act_fn) in zip(axes.flat, activations):
    y_vals = act_fn(x_vals)
    ax.plot(x_vals.numpy(), x_vals.numpy(), 'k--', alpha=0.5, label='y=x')
    ax.plot(x_vals.numpy(), y_vals.numpy(), 'r-', linewidth=2.2, label=name)
    ax.axhline(0, color='gray', linewidth=0.7)
    ax.axvline(0, color='gray', linewidth=0.7)
    ax.grid(True, linestyle=':', alpha=0.6)
    ax.set_ylim(-3.2, 3.2)
    ax.set_title(name, fontweight='bold')
    ax.legend(loc='lower right')

plt.tight_layout()
plt.show()

## 3. Affine Projections with `nn.Linear` & The Callable Rule
`nn.Linear(in_features, out_features)` computes $\mathbf{y} = \mathbf{x}\mathbf{W}^T + \mathbf{b}$.
**Critical Rule:** Always invoke `model(x)` instead of `model.forward(x)` to ensure that pre/post forward hooks execute correctly.

In [ ]:
# 1 input feature -> 1 output feature
linear_model = nn.Linear(1, 1)

print("Initial Weight:", linear_model.weight)
print("Initial Bias:  ", linear_model.bias)

# Forward call using callable syntax
sample_pred = linear_model(t_un)
print("Sample prediction shape:", sample_pred.shape)

### 3.1 Training the Linear Benchmark

In [ ]:
optimizer = optim.SGD(linear_model.parameters(), lr=1e-2)
loss_fn = nn.MSELoss()

for epoch in range(1, 3001):
    t_p = linear_model(t_un)
    loss = loss_fn(t_p, t_c)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

print(f"Final Linear MSE Loss: {loss.item():.4f}")
print(f"Trained weight: {linear_model.weight.item():.4f}, bias: {linear_model.bias.item():.4f}")

## 4. Assembling Deep Architectures with `nn.Sequential`
We construct a genuine multi-layer neural network with 13 hidden units and $\tanh$ non-linearity:
$$[N, 1] \xrightarrow{\text{Linear}(1, 13)} [N, 13] \xrightarrow{\text{Tanh}} [N, 13] \xrightarrow{\text{Linear}(13, 1)} [N, 1]$$

In [ ]:
seq_model = nn.Sequential(
    nn.Linear(1, 13),
    nn.Tanh(),
    nn.Linear(13, 1)
)

print(seq_model)

## 5. Parameter Inspection & Layer Naming with `OrderedDict`
We inspect every trainable parameter tensor and name layers semantically.

In [ ]:
total_params = 0
for name, param in seq_model.named_parameters():
    print(f"{name:15s} | Shape: {str(param.shape):20s} | Parameters: {param.numel()}")
    total_params += param.numel()

print(f"\nTotal Trainable Parameters: {total_params}")

# Semantic naming using OrderedDict
named_seq_model = nn.Sequential(OrderedDict([
    ('hidden_linear', nn.Linear(1, 13)),
    ('hidden_activation', nn.Tanh()),
    ('output_linear', nn.Linear(13, 1))
]))

print("\nNamed layers in model:")
for name, param in named_seq_model.named_parameters():
    print(f"  {name}")

## 6. Training Dynamics: Adaptive Optimization with Adam
We train `seq_model` using Adam (`lr=1e-2`) and observe significant loss reduction compared to the linear model.

In [ ]:
optimizer = optim.Adam(seq_model.parameters(), lr=1e-2)
loss_fn = nn.MSELoss()

loss_history = []
for epoch in range(1, 5001):
    t_p = seq_model(t_un)
    loss = loss_fn(t_p, t_c)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    loss_history.append(loss.item())
    if epoch % 1000 == 0:
        print(f"Epoch {epoch:5d} | Loss: {loss.item():.4f}")

print(f"\nFinal Neural Network Loss: {loss.item():.4f} (vs. Linear Benchmark: 2.9400)")

## 7. Visualizing Non-Linear Interpolation & Expressivity
We pass a continuous temperature range ($[20, 90]^\circ\text{F}$) through our trained network to visualize the smooth learned manifold.

In [ ]:
t_range = torch.arange(20.0, 90.0, 0.5).unsqueeze(1)

with torch.no_grad():
    pred_range = seq_model(0.1 * t_range)
    pred_data = seq_model(t_un)

plt.figure(figsize=(9, 5.5))
plt.plot(t_range.numpy(), pred_range.numpy(), 'c-', linewidth=2.4, label='Neural Network Fit (Continuous)')
plt.scatter(t_u.numpy(), t_c.numpy(), color='blue', s=60, label='Actual Data (Celsius)')
plt.scatter(t_u.numpy(), pred_data.numpy(), color='black', marker='x', s=80, linewidth=2, label='Model Predictions on Data')

plt.xlabel('Fahrenheit (t_u)', fontweight='bold')
plt.ylabel('Celsius (t_c)', fontweight='bold')
plt.title('Non-Linear Neural Network Temperature Calibration', fontweight='bold')
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend()
plt.show()

## 8. Advanced OOP Architecture: Subclassing `nn.Module` & Residual Blocks
When networks require non-sequential topologies (residual connections, branching heads, recurrent loops), we inherit from `nn.Module` directly.

In [ ]:
class CustomMLP(nn.Module):
    def __init__(self, in_features=1, hidden_dim=13, out_features=1):
        super().__init__()
        self.hidden = nn.Linear(in_features, hidden_dim)
        self.activation = nn.Tanh()
        self.output = nn.Linear(hidden_dim, out_features)
        
    def forward(self, x):
        h = self.activation(self.hidden(x))
        out = self.output(h)
        return out

class ResidualBlock(nn.Module):
    """Demonstrates skip connections: y = x + f(x)"""
    def __init__(self, dim):
        super().__init__()
        self.linear = nn.Linear(dim, dim)
        self.act = nn.ReLU()
        
    def forward(self, x):
        return x + self.act(self.linear(x))

mlp = CustomMLP()
res = ResidualBlock(13)

dummy_in = torch.randn(4, 1)
mlp_out = mlp(dummy_in)
res_out = res(torch.randn(4, 13))

print(f"CustomMLP output shape:       {mlp_out.shape}")
print(f"ResidualBlock output shape:   {res_out.shape}")